In [2]:
from pathlib import Path

SOURCE_PATH = Path(r"E:\laba_results\clean_final.parquet")

print("Файл существует:", SOURCE_PATH.exists())

if SOURCE_PATH.exists():
    connection = duckdb.connect()
    schema = connection.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [str(SOURCE_PATH)]
    ).fetchdf()

    display(schema)
    connection.close()

Файл существует: True


,column_name,column_type,null,key,default,extra
0,key,VARCHAR,YES,None,None,None
1,fare_amount,DOUBLE,YES,None,None,None
2,pickup_datetime,TIMESTAMP,YES,None,None,None
3,pickup_longitude,DOUBLE,YES,None,None,None
4,pickup_latitude,DOUBLE,YES,None,None,None
5,dropoff_longitude,DOUBLE,YES,None,None,None
6,dropoff_latitude,DOUBLE,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None


In [4]:
connection = duckdb.connect()

In [5]:
# Проверяем количество строк и уникальных значений

stats = connection.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT key) AS unique_keys,
        COUNT(DISTINCT passenger_count) AS passenger_categories,
        COUNT(DISTINCT pickup_longitude) AS pickup_longitudes,
        COUNT(DISTINCT pickup_latitude) AS pickup_latitudes
    FROM read_parquet(?)
""", [str(SOURCE_PATH)]).fetchdf()

display(stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_keys,passenger_categories,pickup_longitudes,pickup_latitudes
0,55418645,55418644,14,348884,350016


In [6]:
duplicate_keys = connection.execute("""
    SELECT
        key,
        COUNT(*) AS occurrences
    FROM read_parquet(?)
    GROUP BY key
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 10
""", [str(SOURCE_PATH)]).fetchdf()

display(duplicate_keys)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,key,occurrences
0,2014-12-12 11:33:00.00000015,2


In [7]:
passenger_stats = connection.execute("""
    SELECT
        passenger_count,
        COUNT(*) AS trips_count
    FROM read_parquet(?)
    GROUP BY passenger_count
    ORDER BY passenger_count
""", [str(SOURCE_PATH)]).fetchdf()

display(passenger_stats)

,passenger_count,trips_count
0,0,195415
1,1,38334033
2,2,8174602
3,3,2432455
4,4,1178741
5,5,3928864
6,6,1174420
7,7,15
8,8,9
9,9,23


In [8]:
from pathlib import Path
import sqlite3

DB_PATH = Path(r"E:\laba_results\taxi.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
CREATE TABLE IF NOT EXISTS passenger_categories (
    passenger_count INTEGER PRIMARY KEY
);

CREATE TABLE IF NOT EXISTS trips (
    key TEXT PRIMARY KEY,
    fare_amount REAL,
    pickup_datetime TEXT,
    pickup_longitude REAL,
    pickup_latitude REAL,
    dropoff_longitude REAL,
    dropoff_latitude REAL,
    passenger_count INTEGER,
    FOREIGN KEY (passenger_count)
        REFERENCES passenger_categories(passenger_count)
);
""")

conn.commit()

print("База данных:", DB_PATH)
print("Таблицы созданы:")
print(conn.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""").fetchall())

База данных: E:\laba_results\taxi.db
Таблицы созданы:
[('passenger_categories',), ('trips',)]


In [9]:
from pathlib import Path
import sqlite3
import time
import duckdb

SOURCE_PATH = Path(r"E:\laba_results\clean_final.parquet")
DB_PATH = Path(r"E:\laba_results\taxi.db")
BATCH_SIZE = 50_000

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"Не найден файл: {SOURCE_PATH}")

DB_PATH.parent.mkdir(parents=True, exist_ok=True)

# Подключение к Parquet
duck = duckdb.connect()
duck.execute("SET memory_limit = '3GB'")
duck.execute("SET threads = 2")

# Проверяем исходные данные
source_rows, unique_keys, null_keys = duck.execute("""
    SELECT
        COUNT(*),
        COUNT(DISTINCT key),
        COUNT(*) FILTER (WHERE key IS NULL)
    FROM read_parquet(?)
""", [str(SOURCE_PATH)]).fetchone()

print(f"Строк в источнике: {source_rows:,}")
print(f"Уникальных key: {unique_keys:,}")
print(f"Пустых key: {null_keys:,}")

if null_keys:
    duck.close()
    raise ValueError("Обнаружены пустые key. Загрузка остановлена.")

# Создаём SQLite и таблицы
db = sqlite3.connect(DB_PATH)
db.execute("PRAGMA foreign_keys = ON")
db.execute("PRAGMA synchronous = NORMAL")

db.executescript("""
CREATE TABLE IF NOT EXISTS passenger_categories (
    passenger_count INTEGER PRIMARY KEY
);

CREATE TABLE IF NOT EXISTS trips (
    key TEXT PRIMARY KEY,
    fare_amount REAL,
    pickup_datetime TEXT,
    pickup_longitude REAL,
    pickup_latitude REAL,
    dropoff_longitude REAL,
    dropoff_latitude REAL,
    passenger_count INTEGER,
    FOREIGN KEY (passenger_count)
        REFERENCES passenger_categories(passenger_count)
);
""")
db.commit()

try:
    # Очищаем таблицы для чистого повторного запуска
    with db:
        db.execute("DELETE FROM trips")
        db.execute("DELETE FROM passenger_categories")

    # Заполняем справочник
    categories = duck.execute("""
        SELECT DISTINCT passenger_count
        FROM read_parquet(?)
        WHERE passenger_count IS NOT NULL
    """, [str(SOURCE_PATH)]).fetchall()

    with db:
        db.executemany(
            "INSERT INTO passenger_categories VALUES (?)",
            categories
        )

    # Последовательно загружаем данные пакетами
    cursor = duck.execute("""
        SELECT
            CAST(key AS TEXT),
            fare_amount,
            CAST(pickup_datetime AS TEXT),
            pickup_longitude,
            pickup_latitude,
            dropoff_longitude,
            dropoff_latitude,
            passenger_count
        FROM read_parquet(?)
    """, [str(SOURCE_PATH)])

    insert_sql = """
        INSERT INTO trips VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(key) DO NOTHING
    """

    started = time.perf_counter()
    processed = 0
    inserted = 0

    while True:
        batch = cursor.fetchmany(BATCH_SIZE)
        if not batch:
            break

        with db:
            before = db.total_changes
            db.executemany(insert_sql, batch)
            inserted += db.total_changes - before

        processed += len(batch)

        if processed % 500_000 == 0:
            print(f"Обработано: {processed:,} из {source_rows:,}")

    elapsed = time.perf_counter() - started

    # Проверяем результат
    actual_rows = db.execute(
        "SELECT COUNT(*) FROM trips"
    ).fetchone()[0]

    actual_unique = db.execute(
        "SELECT COUNT(DISTINCT key) FROM trips"
    ).fetchone()[0]

    print("\n--- ИТОГ ---")
    print(f"Обработано строк: {processed:,}")
    print(f"Загружено строк: {inserted:,}")
    print(f"Строк в SQLite: {actual_rows:,}")
    print(f"Уникальных ключей в SQLite: {actual_unique:,}")
    print(f"Время загрузки: {elapsed:.2f} сек.")

    if actual_rows != unique_keys or actual_unique != actual_rows:
        raise RuntimeError("Проверка количества строк не пройдена.")

    print("Проверка количества строк и ключей пройдена.")

finally:
    db.close()
    duck.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Строк в источнике: 55,418,645
Уникальных key: 55,418,644
Пустых key: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Обработано: 500,000 из 55,418,645
Обработано: 1,000,000 из 55,418,645
Обработано: 1,500,000 из 55,418,645
Обработано: 2,000,000 из 55,418,645
Обработано: 2,500,000 из 55,418,645
Обработано: 3,000,000 из 55,418,645
Обработано: 3,500,000 из 55,418,645
Обработано: 4,000,000 из 55,418,645
Обработано: 4,500,000 из 55,418,645
Обработано: 5,000,000 из 55,418,645
Обработано: 5,500,000 из 55,418,645
Обработано: 6,000,000 из 55,418,645
Обработано: 6,500,000 из 55,418,645
Обработано: 7,000,000 из 55,418,645
Обработано: 7,500,000 из 55,418,645
Обработано: 8,000,000 из 55,418,645
Обработано: 8,500,000 из 55,418,645
Обработано: 9,000,000 из 55,418,645
Обработано: 9,500,000 из 55,418,645
Обработано: 10,000,000 из 55,418,645
Обработано: 10,500,000 из 55,418,645
Обработано: 11,000,000 из 55,418,645
Обработано: 11,500,000 из 55,418,645
Обработано: 12,000,000 из 55,418,645
Обработано: 12,500,000 из 55,418,645
Обработано: 13,000,000 из 55,418,645
Обработано: 13,500,000 из 55,418,645
Обработано: 14,000,000

In [10]:
import sqlite3
from pathlib import Path

DB_PATH = Path(r"E:\laba_results\taxi.db")

db = sqlite3.connect(DB_PATH)
db.execute("PRAGMA foreign_keys = ON")

# Список таблиц
tables = db.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""").fetchall()

print("Таблицы в базе:")
for table in tables:
    print("-", table[0])

# Количество записей
for table_name in ["trips", "passenger_categories"]:
    count = db.execute(
        f"SELECT COUNT(*) FROM {table_name}"
    ).fetchone()[0]

    print(f"{table_name}: {count:,} строк")

# Проверка внешних ключей
errors = db.execute("PRAGMA foreign_key_check").fetchall()
print("Ошибки внешних ключей:", len(errors))

db.close()

Таблицы в базе:
- passenger_categories
- trips
trips: 55,418,644 строк
passenger_categories: 14 строк
Ошибки внешних ключей: 0


In [11]:
import sqlite3
import time

DB_PATH = r"E:\laba_results\taxi.db"

db = sqlite3.connect(DB_PATH)

queries = {
    "Количество поездок": """
        SELECT COUNT(*) FROM trips
    """,

    "Средняя стоимость по числу пассажиров": """
        SELECT passenger_count,
               COUNT(*) AS trips_count,
               AVG(fare_amount) AS average_fare
        FROM trips
        GROUP BY passenger_count
        ORDER BY passenger_count
    """,

    "Поездки за выбранный период": """
        SELECT COUNT(*)
        FROM trips
        WHERE pickup_datetime >= '2010-01-01'
          AND pickup_datetime < '2011-01-01'
    """
}

for name, sql in queries.items():
    start = time.perf_counter()
    result = db.execute(sql).fetchall()
    elapsed = time.perf_counter() - start

    print(f"\n{name}")
    print(f"Время: {elapsed:.4f} сек.")
    print(f"Результат: {result[:10]}")

db.close()


Количество поездок
Время: 3.6857 сек.
Результат: [(55418644,)]

Средняя стоимость по числу пассажиров
Время: 53.1905 сек.
Результат: [(0, 195415, 9.015453624337948), (1, 38334032, 11.213095579405786), (2, 8174602, 11.82798889536151), (3, 2432455, 11.54222582946036), (4, 1178741, 11.767408641932366), (5, 3928864, 11.209462999991855), (6, 1174420, 12.127515931268201), (7, 15, 31.788666666666664), (8, 9, 29.981111111111108), (9, 23, 36.99304347826087)]

Поездки за выбранный период
Время: 18.2653 сек.
Результат: [(8354648,)]


In [12]:
import sqlite3
import time

DB_PATH = r"E:\laba_results\taxi.db"

db = sqlite3.connect(DB_PATH)

start = time.perf_counter()

db.execute("""
    CREATE INDEX IF NOT EXISTS idx_trips_pickup_datetime
    ON trips(pickup_datetime)
""")

db.commit()

index_time = time.perf_counter() - start

print(f"Время создания индекса: {index_time:.2f} сек.")

# Повторяем выборку за 2010 год
start = time.perf_counter()

result = db.execute("""
    SELECT COUNT(*)
    FROM trips
    WHERE pickup_datetime >= '2010-01-01'
      AND pickup_datetime < '2011-01-01'
""").fetchone()

query_time = time.perf_counter() - start

print(f"Количество поездок: {result[0]:,}")
print(f"Время выборки после индекса: {query_time:.4f} сек.")

db.close()

Время создания индекса: 69.36 сек.
Количество поездок: 8,354,648
Время выборки после индекса: 0.8585 сек.


In [13]:
from pathlib import Path

db_path = Path(r"E:\laba_results\taxi.db")

size_bytes = db_path.stat().st_size
size_mb = size_bytes / (1024 ** 2)

print(f"Размер taxi.db: {size_mb:.2f} МБ")
print(f"Размер taxi.db: {size_bytes:,} байт")

Размер taxi.db: 9171.28 МБ
Размер taxi.db: 9,616,781,312 байт
